In [2]:
import numpy as np
import matplotlib.pyplot as plt
import datetime

In [ ]:
import time # Added import for time.sleep
import os   # Added import for os.path.exists

x = 0 # Initialize x outside the loop so it can increment
while x<100:
  # Format x to a 6-digit string with leading zeros
  filename_formatted_x = f'{x:06d}'

  # Construct full file paths
  filepath_img_array = f'/content/drive/MyDrive/train/train/NoisyLR/{filename_formatted_x}.npy'
  filepath_img_array1 = f'/content/drive/MyDrive/train/train/GT/{filename_formatted_x}.npy'

  # Check if both files exist. If not, break the loop.
  #if not os.path.exists(filepath_img_array) or not os.path.exists(filepath_img_array1):
   #   print(f"Stopping loop: File not found for index {filename_formatted_x}")
    #  break

  # 1. Load the .npy file
  # Corrected typo: np.loadf -> np.load
  # Corrected f-string usage
  img_array = np.load(filepath_img_array)
  img_array1 = np.load(filepath_img_array1)

  # 2. Display the image
  # Use cmap='gray' if your image is grayscale
  plt.imshow(img_array1, cmap='gray')
  plt.axis('on') # Hides the pixel coordinate axes
  plt.title(f"GT - {filename_formatted_x}") # Added dynamic title
  plt.show() # Display image_array

  plt.imshow(img_array, cmap='gray')
  plt.axis('on') # Hides the pixel coordinate axes
  plt.title(f"with noise - {filename_formatted_x}") # Added dynamic title
  plt.show() # Display image_array1

  x += 1 # Increment x for the next iteration
  time.sleep(1) # Replaced delay(500) with time.sleep(0.5)

In [ ]:
import os

folder_path = '/content/drive/MyDrive/train/train/GT'
missing_files1 = [000000]

for i in range(3200):
    filename = f'{i:06d}.npy'
    file_path = os.path.join(folder_path, filename)
    if not os.path.exists(file_path):
        missing_files1.append(filename)

if missing_files1:
    print("Missing files found:")
    for missing_file1 in missing_files1:
        print(missing_file1)
else:
    print("No missing files found in the specified range.")

Missing files found:
0


In [ ]:
# ============================================================
# CELL 1 — Mount Drive, imports, your paths
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from itertools import chain
import numpy as np
import os, glob

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --- YOUR PATHS ---
noisy_path = '/content/drive/MyDrive/train/train/NoisyLR'
gt_path    = '/content/drive/MyDrive/train/train/GT'
ckpt_dir   = '/content/drive/MyDrive/train/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda


In [ ]:
# ============================================================
# CELL 2 — RUN THIS FIRST: check your actual data before training
# Tells you file counts, resolutions present, and pixel range,
# so Cell 5's SCALE value is set correctly instead of guessed.
# ============================================================
noisy_files = sorted(glob.glob(os.path.join(noisy_path, "*.npy")))
gt_files    = sorted(glob.glob(os.path.join(gt_path, "*.npy")))
print(f"Noisy files found: {len(noisy_files)}   GT files found: {len(gt_files)}")

shapes_noisy, mins, maxs = set(), [], []
for f in noisy_files[:50]:
    arr = np.load(f)
    shapes_noisy.add(arr.shape)
    mins.append(arr.min()); maxs.append(arr.max())

shapes_gt = set(np.load(f).shape for f in gt_files[:50])

print("Noisy shapes seen (sample of 50):", shapes_noisy)
print("GT shapes seen (sample of 50):", shapes_gt)
print("Noisy dtype:", np.load(noisy_files[0]).dtype)
print(f"Noisy value range in sample: [{min(mins):.3f}, {max(maxs):.3f}]")

Noisy files found: 3200   GT files found: 3200
Noisy shapes seen (sample of 50): {(128, 128)}
GT shapes seen (sample of 50): {(256, 256)}
Noisy dtype: float32
Noisy value range in sample: [-0.080, 1.714]


In [ ]:
# ============================================================
# CELL 3 — Dataset class + split/bucket helpers
# ============================================================
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os, glob

class RestorationDataset(Dataset):
    """Loads paired (noisy_LR, gt_HR) .npy arrays and returns them as tensors."""
    def __init__(self, noisy_dir, gt_dir, filenames, augment=False, scale=1.0):
        self.noisy_dir = noisy_dir
        self.gt_dir = gt_dir
        self.filenames = filenames
        self.augment = augment
        self.scale = scale   # 1.0 for your data — it's already float32, do NOT divide by 255

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fn = self.filenames[idx]
        noisy = np.load(os.path.join(self.noisy_dir, fn)).astype(np.float32) / self.scale
        gt    = np.load(os.path.join(self.gt_dir, fn)).astype(np.float32) / self.scale

        noisy = torch.from_numpy(noisy).unsqueeze(0)   # add channel dim -> (1, H, W)
        gt    = torch.from_numpy(gt).unsqueeze(0)

        if self.augment:
            if torch.rand(1).item() < 0.5:
                noisy, gt = torch.flip(noisy, [2]), torch.flip(gt, [2])
            if torch.rand(1).item() < 0.5:
                noisy, gt = torch.flip(noisy, [1]), torch.flip(gt, [1])
            k = torch.randint(0, 4, (1,)).item()
            noisy, gt = torch.rot90(noisy, k, [1, 2]), torch.rot90(gt, k, [1, 2])

        return noisy.contiguous(), gt.contiguous()


def build_split(noisy_dir, gt_dir, val_frac=0.1, seed=42):
    """Finds filenames present in BOTH folders, then splits into train/val."""
    noisy_set = set(os.path.basename(f) for f in glob.glob(os.path.join(noisy_dir, "*.npy")))
    gt_set    = set(os.path.basename(f) for f in glob.glob(os.path.join(gt_dir, "*.npy")))
    filenames = sorted(noisy_set & gt_set)
    missing = noisy_set.symmetric_difference(gt_set)
    if missing:
        print(f"Warning: {len(missing)} files present in only one folder — ignored.")

    rng = np.random.default_rng(seed)
    shuffled = list(rng.permutation(filenames))
    n_val = max(1, int(len(shuffled) * val_frac))
    return shuffled[n_val:], shuffled[:n_val]


def bucket_by_shape(noisy_dir, filenames):
    """Groups filenames by input resolution so a DataLoader batch never mixes sizes."""
    buckets = {}
    for fn in filenames:
        shape = np.load(os.path.join(noisy_dir, fn)).shape
        buckets.setdefault(shape, []).append(fn)
    return buckets

print("Cell 3 loaded: RestorationDataset, build_split, bucket_by_shape defined.")

Cell 3 loaded: RestorationDataset, build_split, bucket_by_shape defined.


In [ ]:
# ============================================================
# CELL 4 — Model: U-Net with global residual connection
# ============================================================
import torch.nn as nn
import torch.nn.functional as F

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1), nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1), nn.ReLU(inplace=True),
            )
        self.enc1 = conv_block(1, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = conv_block(128, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = conv_block(128, 64)
        self.final_up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.final_conv = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x); p1 = self.pool1(s1)
        s2 = self.enc2(p1); p2 = self.pool2(s2)
        b = self.bottleneck(p2)
        d2 = self.dec2(torch.cat([self.up2(b), s2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1))
        residual = self.final_conv(self.final_up(d1))
        base = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        return base + residual   # network learns the correction, not raw pixels


class SobelGradLoss(nn.Module):
    """Penalizes edge-map differences so restorations stay sharp, not smoothed."""
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[1,0,-1],[2,0,-2],[1,0,-1]], dtype=torch.float32).view(1,1,3,3)
        ky = kx.transpose(2, 3)
        self.register_buffer('kx', kx)
        self.register_buffer('ky', ky)

    def forward(self, pred, target):
        gx_p = F.conv2d(pred, self.kx, padding=1); gy_p = F.conv2d(pred, self.ky, padding=1)
        gx_t = F.conv2d(target, self.kx, padding=1); gy_t = F.conv2d(target, self.ky, padding=1)
        return F.l1_loss(gx_p, gx_t) + F.l1_loss(gy_p, gy_t)


def psnr(pred, target, max_val=1.0):
    mse = F.mse_loss(pred, target)
    return 10 * torch.log10(max_val**2 / mse) if mse > 0 else torch.tensor(float('inf'))

print("Cell 4 loaded: UNet, SobelGradLoss, psnr defined.")

Cell 4 loaded: UNet, SobelGradLoss, psnr defined.


In [ ]:
# ============================================================
# CELL 5 — Config + build data loaders
# Uses noisy_path / gt_path / ckpt_dir / device from Cell 1 — run Cell 1 first.
# ============================================================
from itertools import chain
import torch.optim as optim
noisy_path = '/content/drive/MyDrive/train/train/NoisyLR'
gt_path    = '/content/drive/MyDrive/train/train/GT'
ckpt_dir   = '/content/drive/MyDrive/train/checkpoints'

SCALE = 1.0          # your data is already float32 in [-0.08, 1.71] — do NOT rescale
BATCH_SIZE = 16
NUM_EPOCHS = 30

train_files, val_files = build_split(noisy_path, gt_path, val_frac=0.1)
print(f"train={len(train_files)}  val={len(val_files)}")

def make_loaders(filenames, augment, shuffle, num_workers):
    buckets = bucket_by_shape(noisy_path, filenames)
    loaders = []
    for shape, files in buckets.items():
        ds = RestorationDataset(noisy_path, gt_path, files, augment=augment, scale=SCALE)
        loaders.append(DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                                   num_workers=num_workers, pin_memory=True))
        print(f"  shape {shape}: {len(files)} images")
    return loaders

print("Train buckets:")
train_loaders = make_loaders(train_files, augment=True, shuffle=True, num_workers=4)
print("Val buckets:")
val_loaders = make_loaders(val_files, augment=False, shuffle=False, num_workers=2)

model = UNet().to(device)
l1 = nn.L1Loss()
grad_loss_fn = SobelGradLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler = torch.amp.GradScaler('cuda')

print("Cell 5 done: model, loaders, optimizer ready.")

train=2880  val=320
Train buckets:


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  shape (128, 128): 2880 images
Val buckets:
  shape (128, 128): 320 images
Cell 5 done: model, loaders, optimizer ready.


In [ ]:
# ============================================================
# CELL 6 — Train (checkpoints saved to Drive every epoch)
# ============================================================
best_val_psnr = -float('inf')
n_train_batches = sum(len(l) for l in train_loaders)
n_val_batches = sum(len(l) for l in val_loaders)

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0
    for noisy, gt in chain(*train_loaders):
        noisy, gt = noisy.to(device, non_blocking=True), gt.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            output = model(noisy)
            loss = l1(output, gt) + 0.1 * grad_loss_fn(output, gt)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    scheduler.step()

    model.eval()
    val_psnr_total = 0
    with torch.no_grad():
        for noisy, gt in chain(*val_loaders):
            noisy, gt = noisy.to(device), gt.to(device)
            with torch.amp.autocast('cuda'):
                output = model(noisy)
            val_psnr_total += psnr(output.float().clamp(0, 1), gt.float()).item()
    val_psnr = val_psnr_total / n_val_batches

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Loss: {train_loss/n_train_batches:.4f} "
          f"Val PSNR: {val_psnr:.2f}dB  LR: {scheduler.get_last_lr()[0]:.2e}")

    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(), 'val_psnr': val_psnr},
               os.path.join(ckpt_dir, 'last_checkpoint.pth'))

    if val_psnr > best_val_psnr:
        best_val_psnr = val_psnr
        torch.save(model.state_dict(), os.path.join(ckpt_dir, 'best_model.pth'))
        print(f"  -> new best saved ({val_psnr:.2f}dB)")

print("Done. Best val PSNR:", best_val_psnr)
print("Model saved at:", os.path.join(ckpt_dir, 'best_model.pth'))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch [1/30] Loss: 0.0767 Val PSNR: 24.58dB  LR: 9.97e-05
  -> new best saved (24.58dB)
Epoch [2/30] Loss: 0.0639 Val PSNR: 25.15dB  LR: 9.89e-05
  -> new best saved (25.15dB)
Epoch [3/30] Loss: 0.0615 Val PSNR: 25.23dB  LR: 9.76e-05
  -> new best saved (25.23dB)
Epoch [4/30] Loss: 0.0604 Val PSNR: 25.30dB  LR: 9.57e-05
  -> new best saved (25.30dB)
Epoch [5/30] Loss: 0.0598 Val PSNR: 25.33dB  LR: 9.33e-05
  -> new best saved (25.33dB)
Epoch [6/30] Loss: 0.0593 Val PSNR: 25.36dB  LR: 9.05e-05
  -> new best saved (25.36dB)
Epoch [7/30] Loss: 0.0590 Val PSNR: 25.35dB  LR: 8.72e-05
Epoch [8/30] Loss: 0.0586 Val PSNR: 25.39dB  LR: 8.35e-05
  -> new best saved (25.39dB)
Epoch [9/30] Loss: 0.0585 Val PSNR: 25.51dB  LR: 7.94e-05
  -> new best saved (25.51dB)
Epoch [10/30] Loss: 0.0581 Val PSNR: 25.49dB  LR: 7.50e-05
Epoch [11/30] Loss: 0.0580 Val PSNR: 25.57dB  LR: 7.03e-05
  -> new best saved (25.57dB)
Epoch [12/30] Loss: 0.0578 Val PSNR: 25.58dB  LR: 6.55e-05
  -> new best saved (25.58dB)
E

In [ ]:
# ============================================================
# CELL 9 — Set up model + inspect the new folder
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os, glob, time
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- SET THIS to your folder of 399 noisy images ---
input_folder  = '/content/drive/MyDrive/train/NoisyLR'   # <-- change this
output_folder = '/content/drive/MyDrive/train/restored_output'
os.makedirs(output_folder, exist_ok=True)

ckpt_path = '/content/drive/MyDrive/train/checkpoints/best_model.pth'

# --- Model definition (must match training) ---
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1), nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1), nn.ReLU(inplace=True),
            )
        self.enc1 = conv_block(1, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = conv_block(128, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = conv_block(128, 64)
        self.final_up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.final_conv = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x); p1 = self.pool1(s1)
        s2 = self.enc2(p1); p2 = self.pool2(s2)
        b = self.bottleneck(p2)
        d2 = self.dec2(torch.cat([self.up2(b), s2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1))
        residual = self.final_conv(self.final_up(d1))
        base = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        return base + residual

model = UNet().to(device)
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()
print("Model loaded from:", ckpt_path)

# --- Inspect the folder before running anything ---
all_files = sorted(os.listdir(input_folder))
print(f"\nTotal files: {len(all_files)}")
ext_counts = Counter(os.path.splitext(f)[1].lower() for f in all_files)
print("File types found:", dict(ext_counts))

sample_file = all_files[0]
sample_path = os.path.join(input_folder, sample_file)
if sample_file.lower().endswith('.npy'):
    arr = np.load(sample_path)
    print(f"\nSample '{sample_file}': shape={arr.shape}, dtype={arr.dtype}, range=[{arr.min():.3f}, {arr.max():.3f}]")
else:
    from PIL import Image
    img = Image.open(sample_path)
    print(f"\nSample '{sample_file}': mode={img.mode}, size={img.size}")

Model loaded from: /content/drive/MyDrive/train/checkpoints/best_model.pth

Total files: 400
File types found: {'.npy': 400}

Sample '000000.npy': shape=(128, 128), dtype=float32, range=[0.001, 1.541]


In [ ]:
# ============================================================
# CELL 10 — Run inference on all 399 images, save outputs
# ============================================================
from PIL import Image

def load_noisy_image(path):
    """Loads either .npy (already normalized like training data) or
    a standard image file (uint8, needs /255 normalization)."""
    if path.lower().endswith('.npy'):
        arr = np.load(path).astype(np.float32)   # already in training-like range, no rescale
    else:
        img = Image.open(path).convert('L')       # force grayscale
        arr = np.array(img).astype(np.float32) / 255.0
    return arr

def save_output(arr, out_path_no_ext, save_npy=True, save_png=True):
    if save_npy:
        np.save(out_path_no_ext + '.npy', arr.astype(np.float32))
    if save_png:
        img = Image.fromarray((arr * 255).clip(0, 255).astype(np.uint8))
        img.save(out_path_no_ext + '.png')

print(f"Processing {len(all_files)} images...")
start = time.time()

with torch.no_grad():
    for i, fn in enumerate(all_files):
        in_path = os.path.join(input_folder, fn)
        noisy = load_noisy_image(in_path)

        inp = torch.from_numpy(noisy).unsqueeze(0).unsqueeze(0).to(device)
        with torch.amp.autocast('cuda'):
            pred = model(inp)
        pred = pred.float().clamp(0, 1).squeeze().cpu().numpy()

        out_name = os.path.splitext(fn)[0]
        save_output(pred, os.path.join(output_folder, out_name))

        if (i + 1) % 50 == 0 or (i + 1) == len(all_files):
            print(f"  {i+1}/{len(all_files)} done")

elapsed = time.time() - start
print(f"\nDone. Total time: {elapsed:.1f}s  ({elapsed/len(all_files)*1000:.1f} ms/image avg)")
print(f"Outputs saved to: {output_folder}")

Processing 400 images...
  50/400 done
  100/400 done
  150/400 done
  200/400 done
  250/400 done
  300/400 done
  350/400 done
  400/400 done

Done. Total time: 17.9s  (44.7 ms/image avg)
Outputs saved to: /content/drive/MyDrive/train/restored_output
